<!-- nav -->
[← 7 · Bindings](07_Bindings.ipynb) · [Home](../../README.md) · [8.01 · Python Types →](08_01_Python_Types.ipynb)

# 8 · Sublanguages

Tutorials 3 to 7 built expressions, the terms a language computes values with. Every language also has grammars
that are not its expressions, written in an expression's place or within one: C writes a type name in a cast,
`(uint8_t)200`; TypeScript a type after `as`; Python, TypeScript and SystemVerilog a pattern where names are bound;
SystemVerilog a sequence in an assertion and a constraint in `randomize() with`; C and SystemVerilog a preprocessor
directive among them; TypeScript a regular expression in a literal, `/a+/g`, and Python and C++ one in a string. Each such grammar is a sublanguage, and each is a dialect of its own, of its language's family:
`Python Types`, `Python Patterns` and `Python RegExp`; `TypeScript Types`, `TypeScript Patterns` and
`TypeScript RegExp`; `C Types`, `Cpp Types`, `Cpp RegExp` and `C Preprocessor`; SystemVerilog's `Types`, `Patterns`, `Assertions`, `Constraints` and `Preprocessor`;
and, where C++'s regular expression names a grammar of POSIX's, POSIX's own, `POSIX BRE` and `POSIX ERE`. Its terms have
kinds, are validated, stored and printed as any dialect's, and are held two ways: in its own holder, an expression
whose root is one of its terms, and as subterms of the expression dialect that reads it, such as a cast's type.
No rule folds a sublanguage's kinds: they are modeled and printed, and stay in the residual, while the expressions that hold them fold as they did before.

Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed and defined. `variable` writes a
name in the dialects whose names are the framework's `variable` (tutorial 4), as tutorials 5 and 6 do. Python's notebook also
reads Python's text, with `Python3.Text.FromText`, which only Python can (tutorial 3); here each Python model is built.
Regular expressions are not shown here, but every sublanguage, regular expressions among them, has a page of its
own after this one, which builds its kinds up from the smallest: the Python family's
[8.01 · Python Types](08_01_Python_Types.ipynb), [8.02 · Python Patterns](08_02_Python_Patterns.ipynb) and
[8.03 · Python RegExp](08_03_Python_RegExp.ipynb); the TypeScript family's
[8.04 · TypeScript Types](08_04_TypeScript_Types.ipynb), [8.05 · TypeScript Patterns](08_05_TypeScript_Patterns.ipynb)
and [8.06 · TypeScript RegExp](08_06_TypeScript_RegExp.ipynb); the C family's [8.07 · C Types](08_07_C_Types.ipynb),
[8.08 · C Preprocessor](08_08_C_Preprocessor.ipynb), [8.09 · Cpp Types](08_09_Cpp_Types.ipynb) and
[8.10 · Cpp RegExp](08_10_Cpp_RegExp.ipynb), with POSIX's [8.11 · POSIX BRE](08_11_POSIX_BRE.ipynb) and
[8.12 · POSIX ERE](08_12_POSIX_ERE.ipynb), which a C++ regular expression may hold; and SystemVerilog's
[8.13 · SystemVerilog Types](08_13_SystemVerilog_Types.ipynb),
[8.14 · SystemVerilog Patterns](08_14_SystemVerilog_Patterns.ipynb),
[8.15 · SystemVerilog Assertions](08_15_SystemVerilog_Assertions.ipynb),
[8.16 · SystemVerilog Constraints](08_16_SystemVerilog_Constraints.ipynb) and
[8.17 · SystemVerilog Preprocessor](08_17_SystemVerilog_Preprocessor.ipynb).

In [1]:
import assert from "node:assert/strict";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as PythonPatterns from "@mbse/expressions/Python/Patterns";
import * as PythonTypes from "@mbse/expressions/Python/Types";
import * as TypeScript5 from "@mbse/expressions/TypeScript/TypeScript5";
import * as TypeScriptPatterns from "@mbse/expressions/TypeScript/Patterns";
import * as TypeScriptTypes from "@mbse/expressions/TypeScript/Types";
import * as C from "@mbse/expressions/C/Expressions";
import * as CPreprocessor from "@mbse/expressions/C/Preprocessor";
import * as CTypes from "@mbse/expressions/C/Types";
import * as Cpp from "@mbse/expressions/Cpp/Expressions";
import * as CppTypes from "@mbse/expressions/Cpp/Types";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as SVAssertions from "@mbse/expressions/SystemVerilog/Assertions";
import * as SVConstraints from "@mbse/expressions/SystemVerilog/Constraints";
import * as SVPatterns from "@mbse/expressions/SystemVerilog/Patterns";
import * as SVPreprocessor from "@mbse/expressions/SystemVerilog/Preprocessor";
import * as SVTypes from "@mbse/expressions/SystemVerilog/Types";
import { Symbolics, Terms } from "@mbse/expressions/Framework";
import { dialects } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const amd64 = store.Instances.Expressions.SystemV.AMD64;
const variable = (name: string) => (d: any) => d.variable((v: any) => v.name(name));

/** What `simplify` says of `expression`: its value, a simplification's where its residual is a literal, or why it has
 * none (`NotEvaluated`). */
function refused(simplify: (expression: any) => unknown, expression: any): string {
  try {
    const result: any = simplify(expression);
    return String(typeof result?.evaluate === "function" ? result.evaluate() : result);
  } catch (error) {
    return (error as Error).message;
  }
}

<a id="T08.1"></a>
## 08.1 · A type is another dialect's term

C's cast `(uint8_t)200` is C's `castexpression`: its `operand` is C's term, the literal `200`, and its `type` is a type
name, `C Types`' `typeid`, its specifiers (here one, the name `uint8_t`) and its abstract declarator (none). TypeScript's
`x as string` is TypeScript5's `tsasexpression`, its `typeAnnotation` a term of `TypeScript Types`, the keyword
`string`; SystemVerilog's `int'(x)` is its `castexpression`, its `type` a term of `SystemVerilog Types`. A form reads
each as it reads any term (tutorial 5): its arguments are the cast's terms, and `Terms.dialect_of` says which dialect's
union holds each one:

In [2]:
const uint8 = (t: any) => t.typeid((i: any) => i.specifiers([(s: any) => s.namedtypespecifier(
  (n: any) => n.name((m: any) => m.identifier((d: any) => d.spelling("uint8_t"))))]));
const casts: Record<string, [any, any]> = store.transaction(() => ({
  C: [C.Text, S.C.Expressions.Expression((e: any) => e.root((u: any) => u.castexpression((r: any) => r
    .type(uint8).operand((o: any) => o.literal((l: any) => l.value((v: any) => v.as_scalar(200n)))))).abi([amd64]))],
  TypeScript5: [TypeScript5.Text, S.TypeScript.TypeScript5.Expression((e: any) => e.root(
    (u: any) => u.tsasexpression((r: any) => r
      .expression((x: any) => x.identifier((i: any) => i.name("x")))
      .typeAnnotation((t: any) => t.tsstringkeyword((k: any) => k)))))],
  SystemVerilog: [SystemVerilog.Text, S.SystemVerilog.Expressions.Expression((e: any) => e.root(
    (u: any) => u.castexpression((r: any) => r
      .type((t: any) => t.integeratomtype((a: any) => a.keyword("int")))
      .value((v: any) => v.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("x"))))))))],
}));
for (const [language, [text, cast]] of Object.entries(casts)) {
  const form = Terms.form(Terms.term(cast));
  console.log(language.padEnd(14), text.ToText(cast).padEnd(17), form.kind);
  for (const argument of form.arguments) {
    console.log(" ".repeat(15), Terms.kindOf(argument)[0].padEnd(17), Terms.dialectOf(argument));
  }
}
assert.deepEqual(Object.values(casts).map(([text, cast]) => text.ToText(cast)), ["(uint8_t)200", "x as string", "int'(x)"]);

C              (uint8_t)200      castexpression


                typeid            Expressions.C.Types.Term


                literal           Expressions.C.Expressions.Term


TypeScript5    x as string       tsasexpression


                identifier        Expressions.TypeScript.TypeScript5.Term


                tsstringkeyword   Expressions.TypeScript.Types.Term


SystemVerilog  int'(x)           castexpression


                integeratomtype   Expressions.SystemVerilog.Types.Term


                nameexpression    Expressions.SystemVerilog.Expressions.Term


Each host reads its sublanguages' terms through the dialects it reads, `reads()`: C reads `C Types`' and
`C Preprocessor`'s, TypeScript5 `TypeScript Types`', `TypeScript Patterns`' and
`TypeScript RegExp`'s, SystemVerilog its five sublanguages'.
`owner` names the dialect whose kind a term is. So the host validates, infers and simplifies a cast with its type in
it, as it did before the types were dialects of their own: C's rules read the type to convert `200`, and infers
the cast's domain from it, the ABI's `uint8_t`; SystemVerilog's converts `x` to an `int`; TypeScript5 folds no `as`, as tutorial 3 said it folds its literals alone:

In [3]:
const hosts: Record<string, [any, any]> = {
  C: [C.Expressions.DIALECT, C.Simplifiers.OfAny],
  TypeScript5: [TypeScript5.Expressions.DIALECT, TypeScript5.Simplifiers.OfAny],
  SystemVerilog: [SystemVerilog.Expressions.DIALECT, SystemVerilog.Simplifiers.OfAny],
};
for (const [language, [, cast]] of Object.entries(casts)) {
  const [dialect, simplify] = hosts[language]!;
  const type = Terms.form(Terms.term(cast)).arguments.find((a: any) => Terms.dialectOf(a) !== dialect.union);
  console.log(language.padEnd(14), "reads", dialect.reads().map((d: any) => d.name()).join(", "));
  console.log(" ".repeat(14), "its type's owner:", dialect.owner(type).name(),
    "| problems:", dialect.validate(cast, ["x"]).join(", ") || "none",
    "| x = 5:", refused((c) => simplify(c, { x: 5n }), cast));
}
console.log("C infers the cast's domain:", C.Expressions.DIALECT.infer(casts.C![1]).name());

C              reads C Types, C Preprocessor


               its type's owner: C Types | problems: none | x = 5: the root is not a literal: its value has no literal


TypeScript5    reads TypeScript Types, TypeScript Patterns, TypeScript RegExp


               its type's owner: TypeScript Types | problems: none | x = 5: the root is not a literal: TypeScript5 tsasexpression is not simplified


SystemVerilog  reads SystemVerilog Types, SystemVerilog Patterns, SystemVerilog Assertions, SystemVerilog Constraints, SystemVerilog Preprocessor


               its type's owner: SystemVerilog Types | problems: none | x = 5: 32'sb00000000000000000000000000000101


C infers the cast's domain: uint8_t


<a id="T08.2"></a>
## 08.2 · A binding target is a pattern

Where TypeScript binds names by destructuring, it writes a pattern: the arrow `([a, b = 1]) => a + b` takes one
parameter, the array pattern `[a, b = 1]`, a term of `TypeScript Patterns` (its `arraypattern`, whose second element is
an `assignmentpattern`, a name and its default). SystemVerilog's `matches` compares a value with a pattern of
`SystemVerilog Patterns`, here the structure pattern `'{.a, .b}`, whose variables it binds. A pattern binds names, but
binds them through what holds it: none of its kinds is a binder of its own (tutorial 7). Each pattern dialect gives the
names a pattern binds, TypeScript's `boundNames` and SystemVerilog's `boundNames`; the arrow binds its parameters'
names in its body, and a conditional binds its `matches`' variables in its consequence, so the only free name left is
`v`, the value matched:

In [4]:
const [arrow, matches]: any[] = store.transaction(() => [
  S.TypeScript.TypeScript5.Expression((e: any) => e.root((u: any) => u.arrowfunctionexpression((a: any) => a
    .params([(p: any) => p.arraypattern((r: any) => r.elements([
      (d: any) => d.identifier((i: any) => i.name("a")),
      (d: any) => d.assignmentpattern((s: any) => s.left((d: any) => d.identifier((i: any) => i.name("b"))).right(
        (o: any) => o.literal((l: any) => l.value((v: any) => v.as_scalar(1)).text("1"))))]))])
    .body((b: any) => b.binaryexpression((o: any) => o
      .left((d: any) => d.identifier((i: any) => i.name("a"))).operator("+")
      .right((d: any) => d.identifier((i: any) => i.name("b")))))))),
  S.SystemVerilog.Expressions.Expression((e: any) => e.root((u: any) => u.conditionalexpression((c: any) => c
    .condition((d: any) => d.matchesexpression((m: any) => m
      .value((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("v")))))
      .pattern((p: any) => p.structurepattern((s: any) => s.items([
        (i: any) => i.variablepattern((r: any) => r.name((n: any) => n.spelling("a"))),
        (i: any) => i.variablepattern((r: any) => r.name((n: any) => n.spelling("b")))])))))
    .consequence((d: any) => d.binaryexpression((b: any) => b
      .left((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("a")))))
      .operator("+")
      .right((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("b")))))))
    .alternative((d: any) => d.literal((l: any) => l.value((v: any) => v.as_scalar(0n))))))),
]);
const parameter = Terms.form(Terms.term(arrow)).arguments.at(-1); // after Let's empty subterms and its body: its one parameter
const pattern = Terms.form(Terms.form(Terms.term(matches)).arguments[0]).arguments[1]; // the condition's pattern
for (const [text, dialect, expression, target, names] of [
  [TypeScript5.Text, TypeScript5.Expressions.DIALECT, arrow, parameter, TypeScriptPatterns.Expressions.boundNames],
  [SystemVerilog.Text, SystemVerilog.Expressions.DIALECT, matches, pattern, SVPatterns.Expressions.boundNames],
] as [any, any, any, any, (term: any) => Set<string>][]) {
  console.log(text.ToText(expression));
  console.log(" ", text.ToText(target).padEnd(9), Terms.kindOf(target)[0], "of", Terms.dialectOf(target));
  console.log("  binds", [...names(target)].sort().join(", "), "| free:",
    [...Symbolics.free(expression, dialect)].sort().join(", ") || "none");
}
assert.equal(TypeScript5.Text.ToText(arrow), "([a, b = 1]) => a + b");
assert.equal(Symbolics.free(arrow, TypeScript5.Expressions.DIALECT).size, 0);
assert.equal(SystemVerilog.Text.ToText(matches), "v matches '{.a, .b} ? a + b : 0");
assert.deepEqual([...Symbolics.free(matches, SystemVerilog.Expressions.DIALECT)], ["v"]);

([a, b = 1]) => a + b


  [a, b = 1] arraypattern of Expressions.TypeScript.Patterns.Term


  binds a, b | free: none


v matches '{.a, .b} ? a + b : 0


  '{.a, .b} structurepattern of Expressions.SystemVerilog.Patterns.Term


  binds a, b | free: v


Python's patterns are those of its `match` statement, `Python Patterns`: `case [a, *rest]:` matches a sequence and binds
`a` and `rest`. A `match` is a statement, which no expression holds, so only the dialect's own holder holds a pattern
here (mbse-programs' `case` holds one too). Python's other binding targets are its expressions as `ast` has them, not
patterns: a comprehension's target, `a, b` in `[a + b for a, b in pairs]`, is Python3's `tuple` of `variable`s, held
by its `comprehension` helper (tutorial 7):

In [5]:
const [sequence, comprehension]: any[] = store.transaction(() => [
  S.Python.Patterns.Expression((e: any) => e.root((u: any) => u.matchsequence((s: any) => s.delimiters("[]").patterns([
    (p: any) => p.matchas((m: any) => m.name("a")), (p: any) => p.matchstar((m: any) => m.name("rest"))])))),
  S.Python.Python3.Expression((e: any) => e.root((u: any) => u.listcomp((c: any) => c
    .body((d: any) => d.binop((b: any) => b.left(variable("a")).op("+").right(variable("b"))))
    .generators([(g: any) => g.target((t: any) => t.tuple((p: any) => p.elts([variable("a"), variable("b")])))
      .iter(variable("pairs"))])))),
]);
console.log(PythonPatterns.Text.ToText(sequence).padEnd(11), Terms.kindOf(Terms.term(sequence))[0], "of",
  Terms.dialectOf(Terms.term(sequence)));
console.log("  binds", [...PythonPatterns.Expressions.boundNames(Terms.term(sequence))].sort().join(", "));
console.log(Python3.Text.ToText(comprehension), "|", Terms.kindOf(Terms.term(comprehension))[0], "of",
  Terms.dialectOf(Terms.term(comprehension)));
const [generator] = Terms.form(Terms.term(comprehension)).attributes.get("generators") as Terms.Part[]; // a helper (tutorial 6)
const target = generator!.attributes.get("target") as Terms.Part;
console.log(" ", generator!.kind, "| its target:", target.kind, "of", target.size,
  "terms, each", Terms.dialectOf(Terms.form(Terms.term(comprehension)).arguments.at(-1)));
assert.equal(PythonPatterns.Text.ToText(sequence), "[a, *rest]");
assert.deepEqual([...PythonPatterns.Expressions.boundNames(Terms.term(sequence))].sort(), ["a", "rest"]);
assert.equal(Python3.Text.ToText(comprehension), "[a + b for a, b in pairs]");

[a, *rest]  matchsequence of Expressions.Python.Patterns.Term


  binds a, rest


[a + b for a, b in pairs] | listcomp of Expressions.Python.Python3.Term


  comprehension | its target: tuple of 2 terms, each Expressions.Python.Python3.Term


<a id="T08.3"></a>
## 08.3 · Each sublanguage a dialect, with its holder

A sublanguage is a dialect like any other (tutorial 1): a singleton of its own, named with its family
(`C Types`, tag `c_types`), its `Term` a union of its kinds, its kinds' editions recorded as data. And it has a holder
of its own, `S.<Family>.<Sublanguage>.Expression`, which refines the framework's `Expression` as every holder does
(tutorial 3): its `root` is a term of its dialect, so a type, a pattern, a sequence, a constraint or a directive can be
written down, stored and printed alone. A C family's holder names its ABI, as C's does (tutorial 2).

Each sublanguage's `Term` lists, besides its own kinds, what may stand where they do: Python Types' lists all of
Python3's branches (a type expression is any expression), SystemVerilog's sublanguages list its `Expressions`'
branches, C Types' lists the preprocessor's directives (an enumeration's items may be). So `T: list[int] | None = int`, a type parameter of
`Python Types`, holds a type expression of Python3's kinds as its bound, which Python's notebook also reads from its text:

In [6]:
const sublanguages: Record<string, [any, any, any]> = store.transaction(() => ({
  "Python Types": [PythonTypes, Python3, S.Python.Types.Expression((e: any) => e.root((u: any) => u.typevar((t: any) => t.name("T")
    .bound((b: any) => b.binop((o: any) => o
      .left((l: any) => l.subscript((s: any) => s.value(variable("list")).slice(variable("int"))))
      .op("|").right((r: any) => r.constant((k: any) => k.singleton("None").text("None")))))
    .default_value(variable("int")))))],
  "Python Patterns": [PythonPatterns, Python3, sequence],
  "TypeScript Types": [TypeScriptTypes, TypeScript5, S.TypeScript.Types.Expression((e: any) => e.root(
    (u: any) => u.tsuniontype((t: any) => t.types([(k: any) => k.tsstringkeyword((h: any) => h),
                                                    (k: any) => k.tsnumberkeyword((h: any) => h)]))))],
  "TypeScript Patterns": [TypeScriptPatterns, TypeScript5, S.TypeScript.Patterns.Expression((e: any) => e.root(
    (u: any) => u.arraypattern((r: any) => r.elements([
      (d: any) => d.identifier((i: any) => i.name("a")),
      (d: any) => d.restelement((s: any) => s.argument((d: any) => d.identifier((i: any) => i.name("rest"))))]))))],
  "C Types": [CTypes, C, S.C.Types.Expression((e: any) => e.root((u: any) => u.typeid((i: any) => i
    .specifiers([(s: any) => s.primitivetypespecifier((p: any) => p.keyword("unsigned")),
                 (s: any) => s.primitivetypespecifier((p: any) => p.keyword("char"))])
    .declarator((d: any) => d.pointerdeclarator((p: any) => p)))).abi([amd64]))],
  "Cpp Types": [CppTypes, Cpp, S.Cpp.Types.Expression((e: any) => e.root((u: any) => u.typeid((i: any) => i
    .specifiers([(s: any) => s.primitivetypespecifier((p: any) => p.keyword("unsigned")),
                 (s: any) => s.primitivetypespecifier((p: any) => p.keyword("long"))]))).abi([amd64]))],
  "C Preprocessor": [CPreprocessor, C, S.C.Preprocessor.Expression((e: any) => e.root(
    (u: any) => u.definedirective((d: any) => d.name((n: any) => n.spelling("WIDTH")).replacement("8"))).abi([amd64]))],
  "SystemVerilog Types": [SVTypes, SystemVerilog, S.SystemVerilog.Types.Expression((e: any) => e.root(
    (u: any) => u.integervectortype((v: any) => v.keyword("bit").dimensions([(d: any) => d.rangedimension((r: any) => r
      .left((l: any) => l.literal((li: any) => li.value((v: any) => v.as_scalar(3n))))
      .right((l: any) => l.literal((li: any) => li.value((v: any) => v.as_scalar(0n)))))]))))],
  "SystemVerilog Patterns": [SVPatterns, SystemVerilog, S.SystemVerilog.Patterns.Expression((e: any) => e.root(
    (u: any) => u.taggedpattern((t: any) => t.member((m: any) => m.spelling("Valid"))
      .pattern((p: any) => p.variablepattern((r: any) => r.name((n: any) => n.spelling("x")))))))],
  "SystemVerilog Assertions": [SVAssertions, SystemVerilog, S.SystemVerilog.Assertions.Expression((e: any) => e.root(
    (u: any) => u.delaysequence((s: any) => s
      .first((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("a")))))
      .steps([(t: any) => t.delay((d: any) => d.literal((l: any) => l.value((v: any) => v.as_scalar(1n))))
        .sequence((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("b")))))]))))],
  "SystemVerilog Constraints": [SVConstraints, SystemVerilog, S.SystemVerilog.Constraints.Expression((e: any) => e.root(
    (u: any) => u.expressionconstraint((c: any) => c.expression((d: any) => d.insideexpression((i: any) => i
      .value((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("x")))))
      .set([(r: any) => r.valuerange((v: any) => v
        .left((l: any) => l.literal((li: any) => li.value((n: any) => n.as_scalar(0n))))
        .right((l: any) => l.literal((li: any) => li.value((n: any) => n.as_scalar(9n)))))]))))))],
  "SystemVerilog Preprocessor": [SVPreprocessor, SystemVerilog, S.SystemVerilog.Preprocessor.Expression((e: any) => e.root(
    (u: any) => u.definedirective((d: any) => d.name((n: any) => n.spelling("WIDTH")).body("8"))))],
}));
for (const [name, [pkg, , holder]] of Object.entries(sublanguages)) {
  const dialect = pkg.Expressions.DIALECT;
  const schema = store.registered(`Expressions.${dialect.path}.Expression`);
  assert.ok(dialect.name() === name && Terms.isExpression(schema) && Terms.dialectOf(Terms.term(holder)) === dialect.union);
  console.log(name.padEnd(27), dialect.tag.padEnd(27), pkg.Text.ToText(holder).padEnd(26), Terms.kindOf(Terms.term(holder))[0]);
}
assert.deepEqual(Object.values(sublanguages).map(([pkg, , holder]) => pkg.Text.ToText(holder)), [
  "T: list[int] | None = int", "[a, *rest]", "string | number", "[a, ...rest]", "unsigned char*", "unsigned long",
  "#define WIDTH 8", "bit [3:0]", "tagged Valid .x", "a ##1 b", "x inside {[0:9]};", "`define WIDTH 8"]);

Python Types                python_types                T: list[int] | None = int  typevar


Python Patterns             python_patterns             [a, *rest]                 matchsequence


TypeScript Types            typescript_types            string | number            tsuniontype


TypeScript Patterns         typescript_patterns         [a, ...rest]               arraypattern


C Types                     c_types                     unsigned char*             typeid


Cpp Types                   cpp_types                   unsigned long              typeid


... (2 outputs left out)

SystemVerilog Patterns      systemverilog_patterns      tagged Valid .x            taggedpattern


SystemVerilog Assertions    systemverilog_assertions    a ##1 b                    delaysequence


SystemVerilog Constraints   systemverilog_constraints   x inside {[0:9]};          expressionconstraint


SystemVerilog Preprocessor  systemverilog_preprocessor  `define WIDTH 8            definedirective


Its expression dialect holds a sublanguage's terms two ways. A subterm of one of its kinds may be of the sublanguage's
`Term`: a cast's type, an arrow's parameters, a `matches`' pattern, a call's arguments (a sequence or a property may
be one), `randomize() with`'s items. And its own `Term` may list a few of the sublanguage's kinds, where its grammar
lets them stand as expressions: TypeScript5's lists the patterns an assignment's target may be, C's and C++'s list
`typeid` (`sizeof(int)`) and the preprocessor's `defined(X)`, SystemVerilog's the preprocessor's macro usage. Python3's
kinds hold neither: a type expression is Python3's own, and a parameter's annotation, which mbse-programs' function
definitions hold, is the helper `arg`'s; a pattern only `match` holds. The properties that hold such a subterm are read from the
store. And each family's rules fold none of the sublanguage's kinds: each stays in the residual, named by the dialect that owns it:

In [7]:
const simplifiers = new Map<any, (expression: any) => unknown>([
  [Python3, Python3.Simplifiers.OfAny], [TypeScript5, TypeScript5.Simplifiers.OfAny], [C, C.Simplifiers.OfAny],
  [Cpp, Cpp.Simplifiers.OfAny], [SystemVerilog, SystemVerilog.Simplifiers.OfAny]]);
for (const [name, [pkg, family, holder]] of Object.entries(sublanguages)) {
  const [union, own] = [store.registered(pkg.Expressions.DIALECT.union), pkg.Expressions.TAGS as readonly string[]];
  const host = store.registered(family.Expressions.DIALECT.union).branches as any[];
  const properties = host.filter((b) => !own.includes(b.name)).flatMap((b) => [...b.schema.properties.entries()]
    .filter(([, prop]: [string, any]) => prop.schema === union || prop.schema.element === union)
    .map(([p]: [string, any]) => `${b.name}.${p}`));
  const listed = host.filter((b) => own.includes(b.name)).map((b) => b.name);
  console.log(name.padEnd(27), family.Expressions.DIALECT.name());
  console.log(" ".repeat(28), "properties:", properties.join(", ") || "none", "| its kinds listed:", listed.join(", ") || "none");
  console.log(" ".repeat(28), refused(simplifiers.get(family)!, holder));
}

Python Types                Python3


                             properties: none | its kinds listed: none


                             the root is not a literal: Python Types typevar is not simplified


Python Patterns             Python3


                             properties: none | its kinds listed: none


                             the root is not a literal: Python Patterns matchsequence is not simplified


... (26 outputs left out)

                             the root is not a literal: SystemVerilog Constraints expressionconstraint is not simplified


SystemVerilog Preprocessor  SystemVerilog


                             properties: none | its kinds listed: macrousage


                             the root is not a literal: SystemVerilog Preprocessor definedirective is not simplified


<a id="T08.4"></a>
## 08.4 · SystemVerilog's assertions and constraints

SystemVerilog Assertions describe behavior over clock cycles: a sequence, `a ##1 b` (`a`, then `b` one cycle later), a
property built of sequences, `a |=> b` (whenever `a`, `b` in the next cycle), and a property's spec, its clock and when
it is disabled, `@(posedge clk) disable iff (rst)`. Each is a kind of `SystemVerilog Assertions`, held by its holder,
or by a call's arguments, where a named sequence or property is given one; the statement that asserts a property,
`assert property (...)`, is mbse-programs'.
SystemVerilog Constraints say what values a randomized object may take: `x inside {[0:9]}`, an expression held as a
constraint, and `x dist {0 := 1, [1:9] :/ 9}`, a distribution of its values, weighted, each a kind of `SystemVerilog
Constraints`, held by its holder or by `randomize() with`'s items. They are modeled and printed, and not simplified:
SystemVerilog's rules neither simulate clock cycles nor solves constraints:

In [8]:
const [handshake, called, randomized]: any[] = store.transaction(() => [
  S.SystemVerilog.Assertions.Expression((e: any) => e.root((u: any) => u.propertyspec((s: any) => s
    .clock((k: any) => k.events([(v: any) => v.edge("posedge").expression(
      (d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("clk")))))]))
    .disable((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("rst")))))
    .property((p: any) => p.implicationproperty((i: any) => i
      .antecedent((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("a")))))
      .operator("|=>")
      .consequent((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("b")))))))))),
  S.SystemVerilog.Expressions.Expression((e: any) => e.root((u: any) => u.callexpression((c: any) => c
    .callee((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("f")))))
    .arguments([(t: any) => t.delaysequence((s: any) => s
      .first((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("a")))))
      .steps([(p: any) => p.delay((d: any) => d.literal((l: any) => l.value((v: any) => v.as_scalar(1n))))
        .sequence((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("b")))))]))])))),
  S.SystemVerilog.Expressions.Expression((e: any) => e.root((u: any) => u.randomizewithexpression((r: any) => r
    .call((c: any) => c.callexpression((m: any) => m.callee((d: any) => d.memberexpression((b: any) => b
      .value((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("p")))))
      .member((i: any) => i.spelling("randomize")))).arguments([])))
    .items([
      (t: any) => t.expressionconstraint((c: any) => c.expression((d: any) => d.insideexpression((i: any) => i
        .value((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("x")))))
        .set([(r: any) => r.valuerange((v: any) => v.left((l: any) => l.literal((li: any) => li.value((n: any) => n.as_scalar(0n))))
          .right((l: any) => l.literal((li: any) => li.value((n: any) => n.as_scalar(9n)))))])))),
      (t: any) => t.expressionconstraint((c: any) => c.expression((d: any) => d.distexpression((i: any) => i
        .value((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("x")))))
        .items([(w: any) => w.value((l: any) => l.literal((li: any) => li.value((n: any) => n.as_scalar(0n))))
          .operator(":=").weight((l: any) => l.literal((li: any) => li.value((n: any) => n.as_scalar(1n)))),
        (w: any) => w.value((r: any) => r.valuerange((v: any) => v
          .left((l: any) => l.literal((li: any) => li.value((n: any) => n.as_scalar(1n))))
          .right((l: any) => l.literal((li: any) => li.value((n: any) => n.as_scalar(9n))))))
          .operator(":/").weight((l: any) => l.literal((li: any) => li.value((n: any) => n.as_scalar(9n))))]))))])))),
]);
const [dialect, simplify] = [SystemVerilog.Expressions.DIALECT, SystemVerilog.Simplifiers.OfAny];
for (const expression of [handshake, called, randomized]) {
  console.log(SystemVerilog.Text.ToText(expression));
  console.log(" ", Terms.form(Terms.term(expression)).arguments.filter((a: any) => a != null)
    .map((a: any) => `${Terms.kindOf(a)[0]} of ${dialect.owner(a).name()}`).join(", "));
  console.log("  free:", [...Symbolics.free(expression, dialect)].sort().join(", "));
}
console.log(refused(simplify, handshake), "|", refused(simplify, randomized));
assert.deepEqual([handshake, called, randomized].map((e) => SystemVerilog.Text.ToText(e)), [
  "@(posedge clk) disable iff (rst) a |=> b", "f(a ##1 b)",
  "p.randomize() with { x inside {[0:9]}; x dist {0 := 1, [1:9] :/ 9}; }"]);

@(posedge clk) disable iff (rst) a |=> b


  nameexpression of SystemVerilog, implicationproperty of SystemVerilog Assertions, nameexpression of SystemVerilog


  free: a, b, clk, rst


f(a ##1 b)


  nameexpression of SystemVerilog, delaysequence of SystemVerilog Assertions


  free: a, b, f


p.randomize() with { x inside {[0:9]}; x dist {0 := 1, [1:9] :/ 9}; }


  callexpression of SystemVerilog, expressionconstraint of SystemVerilog Constraints, expressionconstraint of SystemVerilog Constraints


  free: p, x


the root is not a literal: SystemVerilog Assertions propertyspec is not simplified | the root is not a literal: SystemVerilog randomizewithexpression is not simplified


<a id="T08.5"></a>
## 08.5 · One printer per family

A family prints its dialects with one printer, as mbse-programs prints a program: a host's printer prints the
sublanguages' terms it holds (08.1 to 08.4), and each sublanguage's `Text.ToText` prints through it. Python's,
TypeScript's and SystemVerilog's sublanguages give their family's `ToText` itself; C's and C++'s give one of their own,
which calls the C family's printer naming their dialect. So a tree that crosses dialects prints as one text, whichever
holder holds it. C's enumeration below is a `C Types` term whose items are its enumerators and a `C Preprocessor`
directive, `#ifdef WIDE`, whose items are `C Types`' enumerators again; the printer writes the directive at its line's
start, and the body's enumerators on lines of their own:

In [9]:
for (const [family, subs] of [[Python3, [PythonTypes, PythonPatterns]], [TypeScript5, [TypeScriptTypes, TypeScriptPatterns]],
  [C, [CTypes, CPreprocessor]], [Cpp, [CppTypes]],
  [SystemVerilog, [SVTypes, SVPatterns, SVAssertions, SVConstraints, SVPreprocessor]]] as [any, any[]][]) {
  console.log(family.Expressions.DIALECT.name().padEnd(14),
    ...subs.map((s) => `${s.Expressions.DIALECT.name()}: ${s.Text.ToText === family.Text.ToText ? "True" : "False"}`));
}
const color = store.transaction(() => S.C.Types.Expression((e: any) => e.root((u: any) => u.typeid((i: any) => i.specifiers([
  (s: any) => s.enumspecifier((n: any) => n.key("enum").name((m: any) => m.identifier((d: any) => d.spelling("Color"))).body(true)
    .enumerators([
      (t: any) => t.enumerator((r: any) => r.name((d: any) => d.spelling("RED"))
        .value((v: any) => v.literal((l: any) => l.value((n: any) => n.as_scalar(1n))))),
      (t: any) => t.ifdefdirective((d: any) => d.name((n: any) => n.spelling("WIDE")).items([
        (t: any) => t.c_types((u: any) => u.enumerator((r: any) => r.name((d: any) => d.spelling("GREEN"))))])),
      (t: any) => t.enumerator((r: any) => r.name((d: any) => d.spelling("BLUE")))]))]))).abi([amd64])));
console.log(CTypes.Text.ToText(color));
const directive = Terms.form(Terms.term(color)).arguments[2];
console.log(Terms.kindOf(directive)[0], "of", Terms.dialectOf(directive), "|", CTypes.Expressions.DIALECT.owner(directive).name());
assert.ok(CTypes.Text.ToText(color) === C.Text.ToText(color));
assert.equal(CTypes.Text.ToText(color), "enum Color {\n    RED = 1,\n#ifdef WIDE\n    GREEN,\n#endif\n    BLUE\n}");

Python3        Python Types: True Python Patterns: True


TypeScript5    TypeScript Types: True TypeScript Patterns: True


C              C Types: False C Preprocessor: False


Cpp            Cpp Types: False


SystemVerilog  SystemVerilog Types: True SystemVerilog Patterns: True SystemVerilog Assertions: True SystemVerilog Constraints: True SystemVerilog Preprocessor: True


enum Color {
    RED = 1,
#ifdef WIDE
    GREEN,
#endif
    BLUE
}


ifdefdirective of Expressions.C.Types.Term | C Preprocessor


## What you learned

- **A sublanguage is a dialect of its own**, of its language's family: its kinds, its `Term`, its singleton named with
  its family, its editions as data. Python's types and patterns, TypeScript's types and patterns, C's and C++'s types
  and their preprocessor, and SystemVerilog's types, patterns, assertions, constraints and preprocessor are twelve; with the regular
  expressions of Python, TypeScript and C++, and POSIX's two that C++ reads, seventeen.
- **A type or a pattern is a term**, of its sublanguage, where its host's grammar puts one: a cast's type, an arrow's
  parameters, a `matches`' pattern, a call's argument, `randomize() with`'s items. The host reads it through the
  dialects it reads, and `owner` names the dialect it is of.
- **Each has a holder of its own**, an expression whose root is one of its terms, so a type, a pattern, a sequence, a
  constraint or a directive is written down and stored alone.
- **A pattern binds through what holds it**: an arrow its parameters, a `matches` its variables, a `case` its captures.
- **None is folded**: each family's rules fold none of a sublanguage's kinds, which stay in the residual, named by the
  dialect that owns them, while a host's own kinds keep folding as they did (C's cast reads its type).
- **One printer per family** prints a tree that crosses its dialects as one text.

Next, a page for each sublanguage, starting with [8.01 · Python Types](08_01_Python_Types.ipynb); after them,
Part II puts expressions to work as constraints, starting with case study 9.

<!-- nav -->
[← 7 · Bindings](07_Bindings.ipynb) · [Home](../../README.md) · [8.01 · Python Types →](08_01_Python_Types.ipynb)